# 1. Setting Up

In [ ]:
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL = "airesearch/wangchanberta-base-att-spm-uncased"
ds = load_dataset("pythainlp/wisesight_sentiment")
tokenizer = AutoTokenizer.from_pretrained(MODEL)
print(ds)

README.md:   0%|          | 0.00/12.1k [00:00<?, ?B/s]

wisesight_sentiment/train-00000-of-00001(…): reconstructing file:   0%|          |  0.00B / 2.58MB            

wisesight_sentiment/train-00000-of-00001(…): downloading bytes:           |  0.00B            

wisesight_sentiment/validation-00000-of-(…): reconstructing file:   0%|          |  0.00B /  286kB            

wisesight_sentiment/validation-00000-of-(…): downloading bytes:           |  0.00B            

wisesight_sentiment/test-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B /  327kB            

... (4 outputs left out)

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/905k [00:00<?, ?B/s]

DatasetDict({
    train: Dataset({
        features: ['texts', 'category'],
        num_rows: 21628
    })
    validation: Dataset({
        features: ['texts', 'category'],
        num_rows: 2404
    })
    test: Dataset({
        features: ['texts', 'category'],
        num_rows: 2671
    })
})


In [ ]:
def preprocess(text: str) -> str:
    return text.lower().replace(" ", "<_>")

sample = "ร้านนี้อร่อยมาก แนะนำเลยครับ Delivery ไวด้วย"
print("ไม่ preprocess:", tokenizer.tokenize(sample))
print("preprocess    :", tokenizer.tokenize(preprocess(sample)))

lengths = [len(tokenizer(preprocess(t))["input_ids"]) for t in ds["train"]["texts"][:5000]]
print("\nความยาว token (percentile 50/90/95/99):", np.percentile(lengths, [50, 90, 95, 99]))
print("ยาวสุด:", max(lengths))

ไม่ preprocess: ['▁ร้านนี้', 'อร่อยมาก', '▁', 'แนะนํา', 'เลยครับ', '▁', 'D', 'e', 'live', 'ry', '▁', 'ไว', 'ด้วย']
preprocess    : ['▁ร้านนี้', 'อร่อยมาก', '<_>', '▁', 'แนะนํา', 'เลยครับ', '<_>', '▁', 'de', 'live', 'ry', '<_>', '▁', 'ไว', 'ด้วย']

ความยาว token (percentile 50/90/95/99): [ 16.    80.1  122.05 280.01]
ยาวสุด: 1092


# Tokenize the data

In [ ]:
MAX_LENGTH = 256
LABEL_NAMES = ds["train"].features["category"].names

def tokenize(batch):
    return tokenizer(
        [preprocess(t) for t in batch["texts"]],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized = ds.map(tokenize, batched=True, remove_columns=["texts"])
tokenized = tokenized.rename_column("category", "labels")
print(tokenized)

truncated = sum(len(ids) == MAX_LENGTH for ids in tokenized["train"]["input_ids"])
print(f"\nถูกตัดท้าย: {truncated} ข้อ ({truncated / len(tokenized['train']):.1%})")

Map:   0%|          | 0/21628 [00:00<?, ? examples/s]

Map:   0%|          | 0/2404 [00:00<?, ? examples/s]

Map:   0%|          | 0/2671 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 21628
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 2404
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 2671
    })
})

ถูกตัดท้าย: 302 ข้อ (1.4%)


In [ ]:
from transformers import DataCollatorWithPadding

collator = DataCollatorWithPadding(tokenizer=tokenizer)

batch = collator([tokenized["train"][i] for i in range(4)])
print({k: v.shape for k, v in batch.items()})
print("ความยาวของแต่ละข้อก่อน pad:", [len(tokenized["train"][i]["input_ids"]) for i in range(4)])

{'labels': torch.Size([4]), 'input_ids': torch.Size([4, 64]), 'attention_mask': torch.Size([4, 64])}
ความยาวของแต่ละข้อก่อน pad: [64, 23, 8, 12]


# Training Process

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification

id2label = dict(enumerate(LABEL_NAMES))
label2id = {name: i for i, name in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(LABEL_NAMES), id2label=id2label, label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

model.safetensors: reconstructing file:   0%|          |  0.00B /  423MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] CamembertForSequenceClassification LOAD REPORT from: airesearch/wangchanberta-base-att-spm-uncased
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="wangchanberta-wisesight",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=2,
    fp16=True,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=collator,
    compute_metrics=compute_metrics,
)
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.892123,0.729762,0.683444,0.501648
2,0.716197,0.656260,0.727537,0.618763
3,0.558917,0.631225,0.740017,0.623948


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2028, training_loss=0.6934511971896922, metrics={'train_runtime': 635.356, 'train_samples_per_second': 102.122, 'train_steps_per_second': 3.192, 'total_flos': 6433622775634080.0, 'train_loss': 0.6934511971896922, 'epoch': 3.0})

# Inference

In [ ]:
import json
import time

import torch

model = trainer.model.eval()
device = model.device

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def predict_one(text):
    enc = tokenizer(preprocess(text), truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
    start = time.perf_counter()
    with torch.no_grad():
        logits = model(**enc).logits
    if device.type == "cuda":
        torch.cuda.synchronize()
    latency = time.perf_counter() - start
    return LABEL_NAMES[logits.argmax(-1).item()], latency, enc["input_ids"].shape[1]

predict_one("ทดสอบ")  # warm-up

def predict_file(in_path, out_path):
    results = []
    for row in load_jsonl(in_path):
        pred, latency, n_tokens = predict_one(row["text"])
        results.append({
            "id": row["id"], "label": row["label"], "pred": pred, "raw": pred,
            "latency": latency, "input_tokens": n_tokens, "output_tokens": 0,
        })
    with open(out_path, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"{out_path}: {len(results)} ข้อ")

predict_file("dev.jsonl", "wangchanberta__finetune__run1_dev.jsonl")
predict_file("test.jsonl", "wangchanberta__finetune__run1_test.jsonl")

wangchanberta__finetune__run1_dev.jsonl: 300 ข้อ
wangchanberta__finetune__run1_test.jsonl: 600 ข้อ


In [ ]:
from google.colab import files

files.download("wangchanberta__finetune__run1_dev.jsonl")
files.download("wangchanberta__finetune__run1_test.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
full_test = [
    {"id": f"test-{i}", "text": t, "label": LABEL_NAMES[c]}
    for i, (t, c) in enumerate(zip(ds["test"]["texts"], ds["test"]["category"]))
]
with open("test_full.jsonl", "w", encoding="utf-8") as f:
    for row in full_test:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

predict_file("test_full.jsonl", "wangchanberta__finetune__run1_test_full.jsonl")
files.download("wangchanberta__finetune__run1_test_full.jsonl")

wangchanberta__finetune__run1_test_full.jsonl: 2671 ข้อ


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>